# Challenge 3 · Gate Surgeon
**Tier 1 - Foundations** · **Estimated time: 20-30 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Three circuits arrive at the clinic with symptoms: wrong states, wrong unitaries, wrong angles. Diagnose the bugs and operate - without changing what each circuit is supposed to do.

### Learning objectives
- Read a circuit and predict the state it prepares.
- Debug common mistakes: wrong qubit, reversed CNOT, stray gate, wrong rotation angle.
- Verify a repair by comparing states and unitaries, not by eye.
- Recognise that many different circuits can be correct.

### Prerequisites
- Challenges 1-2 or equivalent
- Qubit ordering in Qiskit (qubit 0 is the rightmost bit)

### What you will write
| Function | Status |
|---|---|
| `patient_ghz` | required |
| `patient_swap` | required |
| `patient_tilted_pair` | required |
| `patient_readout` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `patient_ghz()` returns a three-qubit circuit preparing (|000>+|111>)/sqrt(2) (up to global phase).
- `patient_swap()` uses only CX gates (at most three) and implements SWAP (up to global phase).
- `patient_tilted_pair(theta)` prepares cos(theta/2)|00> + sin(theta/2)|11> for every theta.
- Non-finite theta is rejected with ValueError; the returned circuits are measurement-free and have the documented width.

**Bonus (optional, never required to pass):**
- `patient_readout()` yields the deterministic bit-string '110' with qubit i measured into clbit i.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
Three circuits (and one bonus patient) arrive at the clinic. Each *runs*, but does the wrong thing. Your job is to **repair them without
changing what they are supposed to do**, and to prove the repair with state vectors and unitaries - not by staring at the drawing.

| Patient | Must become | Constraint |
|---|---|---|
| 1 · `patient_ghz()` | the 3-qubit GHZ state (&#124;000⟩ + &#124;111⟩)/√2 | 3 qubits, gates only |
| 2 · `patient_swap()` | the SWAP unitary on 2 qubits | **only CX gates, at most 3** |
| 3 · `patient_tilted_pair(theta)` | cos(θ/2)&#124;00⟩ + sin(θ/2)&#124;11⟩ **for every θ** | 2 qubits, gates only; non-finite θ raises `ValueError` |
| 4 · `patient_readout()` *(bonus)* | a 3-qubit circuit with 3 classical bits whose measurement gives **`110` with certainty** | qubit *i* is measured into classical bit *i* |

### Function contracts
* "Prepares a state" means: starting from &#124;0…0⟩ the circuit's state vector equals the target **up to a global phase**. Many different circuits are correct.
* Circuits returned for patients 1-3 contain **no measurements and no classical bits**.
* Qiskit's bit order: qubit 0 is the **rightmost** character of a bit-string.
* `patient_tilted_pair` must work for *any* finite real θ (negative, larger than 2π, integers, NumPy floats). `float('nan')` and `inf` must raise `ValueError`.

**Tolerances.** States and unitaries are compared exactly (up to global phase) with absolute tolerance 1e-9.

## Debugging toolkit
* `describe_state(circuit)` lists the non-zero amplitudes of the prepared state.
* `state_fidelity(circuit, target)` is 1.0 exactly when the state matches the target (up to global phase).
* `Operator(circuit)` gives the circuit's unitary matrix, and `Operator.equiv` compares two unitaries up to global phase.

**Rule of the clinic:** trust the state vector, not the picture. A CNOT whose control qubit is |0⟩ silently does nothing.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

In [ ]:
from qiskit.quantum_info import Operator, Statevector
from qiskit_fest.challenges.c03_gate_surgeon import (
    GHZ_TARGET, SWAP_TARGET, buggy_ghz, buggy_readout, buggy_swap, buggy_tilted_pair, describe_state,
    plot_fidelity_scan, state_fidelity, tilted_target)
from qiskit_fest.viz import draw_circuit, plot_distributions

## Patient 1 · The GHZ state  *(fix the code)*
**Chart.** `patient_ghz()` should prepare $(|000\rangle+|111\rangle)/\sqrt2$ - three qubits that are all 0 or all 1. Here is what the broken circuit actually does:

In [ ]:
describe_state(buggy_ghz(), "what the BROKEN patient 1 prepares")
print("fidelity with the GHZ target:", round(state_fidelity(buggy_ghz(), GHZ_TARGET), 3))

*Symptom:* one qubit is stuck at 1, another at 0, and only one is random - the qubits are not correlated at all.
Edit the function below until the state matches the target. Several different circuits are correct.

<details><summary>💡 Hint 1 (subtle)</summary>

In a GHZ state exactly **one** qubit is put into superposition and the other qubits *copy* it. Count how many qubits in the printed state are random, and which are constant.
</details>

<details><summary>💡 Hint 2</summary>

A CNOT acts only when its **control** (the *first* argument of `cx`) is |1⟩. If the control is |0⟩ it does nothing. Is that happening to the CNOTs here? Also: is every gate in the circuit actually needed?
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Pick one qubit for the Hadamard and use that same qubit as the control of both CNOTs (targets: the other two qubits). Remove the stray gate.
</details>

In [ ]:
def patient_ghz() -> QuantumCircuit:
    """Should prepare (|000> + |111>)/sqrt(2).  THIS CIRCUIT IS BROKEN - FIX IT."""
    qc = QuantumCircuit(3)
    qc.h(2)
    qc.cx(1, 0)
    qc.cx(0, 2)
    qc.x(1)
    return qc

In [ ]:
describe_state(patient_ghz(), "your patient 1")
print("fidelity with the GHZ target:", round(state_fidelity(patient_ghz(), GHZ_TARGET), 6))
draw_circuit(patient_ghz(), "Patient 1 after surgery")

In [ ]:
report = run_public_checks(3, globals(), keyword="patient_ghz")

## Patient 2 · The SWAP gate  *(fix the code)*
**Chart.** `patient_swap()` must implement SWAP (exchange two qubits' states) using **only CX gates**. The broken version:

In [ ]:
broken = buggy_swap()
print("broken circuit equals SWAP? ", Operator(broken).equiv(SWAP_TARGET))
for label, name in (("01", "|q1 q0> = |01>"), ("10", "|q1 q0> = |10>")):
    out = Statevector.from_label(label).evolve(broken)
    k = int(np.argmax(np.abs(out.data)))
    print(f"  {name} -> |{format(k, '02b')}>   (a SWAP would give |{label[::-1]}>)")

*Symptom:* a swap must turn |01⟩ into |10⟩ and vice versa; this circuit leaves |01⟩ alone and sends |10⟩ to |11⟩.

<details><summary>💡 Hint 1 (subtle)</summary>

Write down what each CNOT does to the basis states and combine them. Two CNOTs in a row with identical arguments do what?
</details>

<details><summary>💡 Hint 2</summary>

A SWAP can be built from exactly three CNOTs, but not three *identical* ones.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

The three CNOTs alternate their direction: the middle one points the opposite way from the outer two.
</details>

In [ ]:
def patient_swap() -> QuantumCircuit:
    """Should implement SWAP with CX gates only (at most 3).  THIS CIRCUIT IS BROKEN - FIX IT."""
    qc = QuantumCircuit(2)
    qc.cx(0, 1)
    qc.cx(0, 1)
    qc.cx(1, 0)
    return qc

In [ ]:
print("equals SWAP? ", Operator(patient_swap()).equiv(SWAP_TARGET), "| gates used:", dict(patient_swap().count_ops()))
draw_circuit(patient_swap(), "Patient 2 after surgery")

In [ ]:
report = run_public_checks(3, globals(), keyword="patient_swap")

## Patient 3 · A tilted entangled pair  *(fix the code, for every angle)*
**Chart.** `patient_tilted_pair(theta)` should prepare $\cos\frac\theta2|00\rangle+\sin\frac\theta2|11\rangle$ - a Bell-like pair whose "tilt" θ is a parameter.
The scan below plots how close the broken circuit gets for each θ (1.0 would be perfect):

In [ ]:
plot_fidelity_scan(buggy_tilted_pair)

*Symptom:* the fidelity is 1 only for a few special angles. A repair that works for one θ is not a repair - your function must work for **every** finite θ, and must reject `nan`/`inf` with a `ValueError`.

<details><summary>💡 Hint 1 (subtle)</summary>

$R_Y(a)|0\rangle=\cos\frac a2|0\rangle+\sin\frac a2|1\rangle$. Which value of `a` gives the amplitudes in the target? Check the angle you pass to `ry`.
</details>

<details><summary>💡 Hint 2</summary>

After the rotation, the CNOT should *copy* the rotated qubit onto the other one. Which qubit is the control in the broken circuit, and which one was rotated?
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Pass the full `theta` to `ry` on qubit 0 and make qubit 0 the control of the CNOT. At the top of the function, validate `theta` with `np.isfinite`.
</details>

In [ ]:
def patient_tilted_pair(theta: float) -> QuantumCircuit:
    """Should prepare cos(theta/2)|00> + sin(theta/2)|11> for every finite theta.  THIS CIRCUIT IS BROKEN - FIX IT."""
    qc = QuantumCircuit(2)
    qc.ry(theta / 2, 0)
    qc.cx(1, 0)
    return qc

In [ ]:
plot_fidelity_scan(patient_tilted_pair)

In [ ]:
report = run_public_checks(3, globals(), keyword="patient_tilted_pair")

## Bonus · Patient 4: the reversed readout  *(optional)*
`patient_readout()` should build a 3-qubit, 3-classical-bit circuit whose measurement outcome is **`110` with certainty**, measuring qubit *i* into classical bit *i*.
The broken version prints something else. Bit-strings put the **highest** classical bit on the **left**.

<details><summary>💡 Hint 1</summary>

Which qubits must be |1⟩ for the string `110`, if the leftmost character is classical bit 2? And where does the broken circuit send each qubit?
</details>

<details><summary>💡 Hint 2 (explicit)</summary>

Qubits 1 and 2 must be |1⟩, qubit 0 stays |0⟩, and `measure([0, 1, 2], [0, 1, 2])` keeps the identity mapping.
</details>

In [ ]:
def patient_readout() -> QuantumCircuit:
    """Should give '110' with certainty, measuring qubit i into clbit i.  THIS CIRCUIT IS BROKEN - FIX IT."""
    qc = QuantumCircuit(3, 3)
    qc.x(1)
    qc.measure([0, 1, 2], [2, 1, 0])
    return qc

In [ ]:
from qiskit_fest.sim import measured_distribution
plot_distributions({"broken": measured_distribution(buggy_readout()), "yours": measured_distribution(patient_readout())},
                   "Exact outcome of the readout patients (target: 110 with probability 1)")

In [ ]:
report = run_public_checks(3, globals(), keyword="bonus")

## Reflection
1. Your repaired GHZ circuit is probably different from your neighbour's. Why can both be correct, and how would you *prove* that they are?
2. Patient 3 passed a quick test at θ = π but failed in general. What does that teach you about testing a parameterised circuit?
3. A CNOT whose control is |0⟩ "does nothing". How can you detect such a silent bug without drawing the circuit?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 3)